# Train the Dual-Head U-Net on Synthetic Fields

This notebook is the main workflow. It creates polygon-like agricultural fields, trains the model, evaluates region and boundary metrics, and shows predictions.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
from copy import deepcopy
import matplotlib.pyplot as plt

from cfgs import field_segmentation
from src.utils.utils import seed_everything
from src.utils.visualization import show_batch, show_predictions

seed_everything(0)
config = deepcopy(field_segmentation.synthetic_debug)

## Load Data

The synthetic dataset provides RGB images, binary masks, instance IDs, and signed-distance-field targets. Replace `synthetic_debug` with `ftw_dual_head` once the real dataset is placed under `data/ftw`.

In [ ]:
dm = config['datamodule'](**config['data_args'])
train_loader = dm.get_loader()
valid_loader = dm.get_heldout_loader()

batch = next(iter(train_loader))
show_batch(batch, max_items=3);

## Train

The debug config only trains for two epochs so the notebook stays lightweight. Increase `config['trainer_config']['epochs']` or switch to `synthetic_full` for a stronger run.

In [ ]:
trainer = config['trainer_module'](
    config=config,
    log_dir=PROJECT_ROOT / 'Logs',
    train_loader=train_loader,
    eval_loader=valid_loader,
)
history = trainer.train()
history

In [ ]:
eval_result = trainer.evaluate(valid_loader)
eval_result

## Visualize Predictions

In [ ]:
batch = next(iter(valid_loader))
show_predictions(trainer.model, batch, device=trainer.device, max_items=3);

## Checkpoints

The final model weights are saved under `Saved/<experiment_name>/last_model.pth`.